# TrialMatch AI — Interactive Demo

Run the cells top-to-bottom. Edit the patient description in **Step 2** and re-run from there.

## Step 1 — Setup

In [ ]:
import sys, os
# Make sure the trialmatch package root is on the path
REPO_ROOT = os.path.dirname(os.path.abspath('.'))
PKG_ROOT  = os.path.join(REPO_ROOT, 'trialmatch')
if PKG_ROOT not in sys.path:
    sys.path.insert(0, PKG_ROOT)

# Suppress noisy logs — set to logging.INFO to see full pipeline trace
import logging
logging.basicConfig(level=logging.WARNING, format='%(asctime)s  %(levelname)-8s  %(message)s', datefmt='%H:%M:%S')
logging.getLogger('httpx').setLevel(logging.WARNING)

from pipeline.extractor import extract_patient_profile
from pipeline.retriever import retrieve_trials
from pipeline.matcher  import ClaudeMatcher
from pipeline.ranker   import rank_trials
from pipeline.explainer import generate_all_cards

print('✓ Imports OK')

## Step 2 — Patient Description

Edit `PATIENT_TEXT` below, or point `PATIENT_FILE` at one of the sample files.

In [ ]:
# ── Option A: type / paste a description directly ────────────────────────────
PATIENT_TEXT = """
I am a 58-year-old man and my doctor recently told me I have Stage IIIB non-small cell
lung cancer. My EGFR test came back negative. I have not had any chemotherapy or other
cancer treatments yet. I live in Indianapolis, Indiana.
""".strip()

# ── Option B: load from a sample file (comment out Option A first) ───────────
# PATIENT_FILE = 'data/sample_patients/patient_02.txt'
# with open(PATIENT_FILE) as f:
#     PATIENT_TEXT = f.read().strip()

print(PATIENT_TEXT)

## Step 3 — Run the Pipeline

In [ ]:
import time
from IPython.display import display, Markdown, HTML

t0 = time.perf_counter()

# 1. Extract
print('Extracting patient profile...')
profile = extract_patient_profile(PATIENT_TEXT)

# 2. Retrieve
print('Retrieving trials...')
condition = profile.conditions[0] if profile.conditions else PATIENT_TEXT.split()[0]
trials = retrieve_trials(condition, profile=profile)

# 3. Match
print(f'Matching {len(trials)} trials (this takes a few minutes)...')
matcher = ClaudeMatcher()
match_results = matcher.match_trials(profile, trials)

# 4. Rank
ranked = rank_trials(match_results)

# 5. Explain
print('Generating patient-friendly cards...')
cards = generate_all_cards(ranked)

elapsed = time.perf_counter() - t0
print(f'\nDone in {elapsed/60:.1f} min  ({len(cards)} top matches found)')

## Step 4 — Results

In [ ]:
from IPython.display import display, Markdown, HTML

# ── Patient summary ───────────────────────────────────────────────────────────
display(Markdown('### Patient Profile'))
display(HTML(f"""
<table style="border-collapse:collapse; font-family:sans-serif; font-size:14px">
  <tr><td style="padding:4px 12px"><b>Age</b></td><td style="padding:4px 12px">{profile.age}</td></tr>
  <tr><td style="padding:4px 12px"><b>Conditions</b></td><td style="padding:4px 12px">{', '.join(profile.conditions or []) or '—'}</td></tr>
  <tr><td style="padding:4px 12px"><b>Stage</b></td><td style="padding:4px 12px">{profile.stage or '—'}</td></tr>
  <tr><td style="padding:4px 12px"><b>Location</b></td><td style="padding:4px 12px">{profile.location or '—'}</td></tr>
</table>
"""))

# ── Score overview table ──────────────────────────────────────────────────────
display(Markdown(f'### Top {len(cards)} Trial Matches'))

rows = ''.join(
    f"""<tr>
      <td style="padding:6px 12px">{i}</td>
      <td style="padding:6px 12px"><a href="https://clinicaltrials.gov/study/{c['nct_id']}" target="_blank">{c['nct_id']}</a></td>
      <td style="padding:6px 12px; text-align:center">
        <span style="background:{'#d4edda' if c['match_score']>=0.5 else '#fff3cd' if c['match_score']>=0.2 else '#f8d7da'};
               padding:2px 8px; border-radius:4px">{c['match_score']:.2f}</span>
      </td>
      <td style="padding:6px 12px; text-align:center">{c['fk_grade']:.1f}</td>
      <td style="padding:6px 12px; text-align:center; color:{'#856404' if c.get('uncertain_count',0)>0 else 'green'}">
        {'⚠️ ' + str(c['uncertain_count']) + ' uncertain' if c.get('uncertain_count',0)>0 else '✓'}
      </td>
    </tr>"""
    for i, c in enumerate(cards, 1)
)

display(HTML(f"""
<table style="border-collapse:collapse; font-family:sans-serif; font-size:14px; width:100%">
  <thead style="background:#f0f0f0">
    <tr>
      <th style="padding:6px 12px; text-align:left">#</th>
      <th style="padding:6px 12px; text-align:left">NCT ID</th>
      <th style="padding:6px 12px">Match Score</th>
      <th style="padding:6px 12px">Readability (FK)</th>
      <th style="padding:6px 12px">Uncertain Criteria</th>
    </tr>
  </thead>
  <tbody>{rows}</tbody>
</table>
"""))

In [ ]:
# ── Full patient-friendly cards ───────────────────────────────────────────────
display(Markdown('### Patient Cards'))

for i, card in enumerate(cards, 1):
    uncertain_html = (
        f'<div style="background:#fff3cd; border-left:4px solid #ffc107; padding:8px 12px; '
        f'margin-bottom:8px; font-size:13px">'
        f'⚠️ <b>{card["uncertain_count"]} criteria uncertain</b> — physician review recommended</div>'
        if card.get('uncertain_count', 0) > 0 else ''
    )
    display(HTML(f"""
    <div style="border:1px solid #dee2e6; border-radius:8px; padding:16px; margin-bottom:20px;
                font-family:sans-serif">
      <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:8px">
        <span style="font-weight:bold; font-size:16px">Match #{i}</span>
        <span style="font-size:13px; color:#555">
          NCT: <a href="https://clinicaltrials.gov/study/{card['nct_id']}" target="_blank">{card['nct_id']}</a>
          &nbsp;|&nbsp; Score: <b>{card['match_score']:.2f}</b>
          &nbsp;|&nbsp; Readability: FK {card['fk_grade']:.1f}
        </span>
      </div>
      {uncertain_html}
    </div>
    """))
    display(Markdown(card['card_text']))
    display(HTML('<hr style="margin:24px 0">'))

## Step 5 — Save Results (optional)

In [ ]:
import json, pathlib

out_path = pathlib.Path('results/notebook_run.json')
out_path.parent.mkdir(exist_ok=True)
out_path.write_text(json.dumps(cards, indent=2))
print(f'Saved to {out_path.resolve()}')